In [13]:
# %%
# =============================================================================
# balanced_ppo.ipynb
# EGX30 Balanced PPO Portfolio Optimizer — v1_initial
#
# Architecture: Shared-weight MLP encoder + Self-attention + PPO (SB3)
#
# Pipeline:
#   PART 1 — Feature Engineering
#     1.  Imports & configuration
#     2.  Load raw OHLCV CSVs
#     3.  Build aligned OHLCV panels & validity mask
#     4.  Compute 9 per-stock features
#     5.  Load macro time series (CONIA, inflation)
#     6.  Compute EGX30 proxy & market regime signal
#     7.  Load daily risk classifications & build profile mask
#     8.  Trim to usable date range
#     9.  Train/val/test split & normalization
#     10. Build & save tensors + preprocessing artifacts
#     11. Sanity checks
#
#   PART 2 — RL Environment & Training
#     12. Load artifacts
#     13. Architecture: SharedStockEncoder + StockAttention + AttentionExtractor
#     14. PortfolioEnv
#     15. Walk-forward fold index builder
#     16. Callbacks
#     17. Helper functions
#     18. Sanity check (single episode)
#     19. Walk-forward training loop
#     20. Multi-episode evaluation
#     21. Aggregate results & summary
#     22. Signal/cash correlation diagnostic
# =============================================================================

In [14]:
from pathlib import Path
import warnings
import json
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F_torch

from scipy import stats
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    StopTrainingOnNoModelImprovement,
    BaseCallback,
)
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [15]:
# Notebook lives at project root — ROOT is the current directory
ROOT = Path(".").resolve()

OHLCV_DIR          = ROOT / "data/OHLCV"
MACRO_SIGNALS_DIR  = ROOT / "data/macro_signals"
TENSORS_DIR        = ROOT / "data/tensors"
PREPROCESSING_DIR  = ROOT / "data/preprocessing"
MODEL_INPUTS_DIR   = ROOT / "data/model_inputs"
MODEL_VERSIONS_DIR = ROOT / "model_versions"

# Source files
LGBM_PRED_PATH             = MODEL_INPUTS_DIR  / "lgbm_predictions.csv"
DAILY_CLASSIFICATIONS_PATH = MODEL_INPUTS_DIR  / "daily_risk_classifications.csv"
CONIA_DATA_PATH            = MACRO_SIGNALS_DIR / "conia_o_n_rate.xlsx"
INFLATION_DATA_PATH        = MACRO_SIGNALS_DIR / "inflation_rate.xlsx"
MARKET_DATA_PATH           = MACRO_SIGNALS_DIR / "market_data.csv"

for d in [TENSORS_DIR, PREPROCESSING_DIR, MACRO_SIGNALS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [16]:
VERSION      = "v1_initial"
RISK_PROFILE = "balanced"

VERSION_DIR = MODEL_VERSIONS_DIR / VERSION
VERSION_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
try:
    from rrfr_fetcher import get_real_risk_free_rate
    _fetched_rrfr = get_real_risk_free_rate()
    ANNUAL_RISK_FREE_RATE = _fetched_rrfr if _fetched_rrfr is not None else 0.21
    _rrfr_source = "live" if _fetched_rrfr is not None else "fallback (fetch returned None)"
except Exception as _e:
    ANNUAL_RISK_FREE_RATE = 0.21
    _rrfr_source = f"fallback (import/fetch error: {_e})"

print(f"Annual risk-free rate : {ANNUAL_RISK_FREE_RATE:.4%}  [{_rrfr_source}]")

2026-06-24 00:59:11,846 - INFO - CONIA: fresh (cached 2026-06-23)
2026-06-24 00:59:11,849 - INFO - Inflation: fresh (cached 2026-06)


Annual risk-free rate : 6.4572%  [live]


In [18]:
VOLUME_AVG_WINDOW  = 20
RETURN_CLAMP       = 0.25
CORR_WINDOW        = 20
MA_SHORT           = 20
MA_LONG            = 60
CONIA_DELTA_WINDOW = 60

FEATURE_NAMES = [
    "ret_1d",
    "ret_5d",
    "ret_20d",
    "lgbm_pred",
    "gk_vol_20d",
    "skew_20d",
    "kurt_20d",
    "rel_volume",
    "avg_pairwise_corr",
]

TRAIN_END = "2021-12-31"
VAL_END   = "2022-12-31"

In [19]:
ENCODER_INPUT_DIM  = len(FEATURE_NAMES) + 1   # 9 features + 1 current weight = 10
ENCODER_HIDDEN_DIM = 64
ENCODER_OUTPUT_DIM = 64
ATTENTION_HEADS    = 1
ATTENTION_DIM      = ENCODER_OUTPUT_DIM
POLICY_HIDDEN_DIMS = [256, 256]

# Soft masking — False = hard binary mask (only balanced-classified stocks)
#                True  = soft weights (balanced=1.0, conservative=0.5, aggressive=0.2)
USE_SOFT_MASK                  = False
SOFT_MASK_CONSERVATIVE_WEIGHT = 0.5
SOFT_MASK_AGGRESSIVE_WEIGHT   = 0.2

In [ ]:
EPISODE_LENGTH        = 24
REBALANCE_DAYS        = 21
PERIODS_PER_YEAR      = 252 / 21
MAX_CASH_WEIGHT       = 0.30
MAX_STOCK_WEIGHT      = 0.20
MIN_STOCK_WEIGHT      = 0.0
BASE_TRANSACTION_COST = 0.0064
MAX_TRANSACTION_COST  = 0.015
TURNOVER_CAP          = 0.80
N_ENVS                = 4
BASE_TIMESTEPS        = 3_500_000
MAX_EVAL_EPISODES     = 50
N_BOOTSTRAP           = 2000
BOOTSTRAP_BLOCK       = 3
CONFIDENCE_LEVEL      = 0.95

WALK_FORWARD_FOLDS = [
    {"name": "fold_1", "train_end": "2019-12-31", "val_end": "2020-12-31", "test_end": "2021-12-31"},
    {"name": "fold_2", "train_end": "2020-12-31", "val_end": "2021-12-31", "test_end": "2022-12-31"},
    {"name": "fold_3", "train_end": "2021-12-31", "val_end": "2022-12-31", "test_end": "2023-12-31"},
    {"name": "fold_4", "train_end": "2022-12-31", "val_end": "2023-12-31", "test_end": "2025-12-31"},
]

PPO_PARAMS = {
    "learning_rate": 3e-4,
    "n_steps":       2048,
    "batch_size":    64,
    "n_epochs":      10,
    "gamma":         0.995,
    "gae_lambda":    0.95,
    "clip_range":    0.2,
    "ent_coef":      0.01,
    "vf_coef":       0.5,
    "max_grad_norm": 0.5,
    "verbose":       0,
}

REWARD_CONFIGS = {
    "return_weight":    1.0,
    "turnover_penalty": 0.01,
    "downside_penalty": 0.8,
    "drawdown_penalty": 1.0,
    "hhi_penalty":      0.3,
    "cash_penalty":     2.0,
}

PALETTE = {
    "primary":   "#2c7bb6",
    "secondary": "#d7191c",
    "tertiary":  "#fdae61",
    "train":     "#2c7bb6",
    "val":       "#fdae61",
    "test":      "#2ca25f",
    "alert":     "#d7191c",
}

print(f"Configuration loaded.")
print(f"  VERSION        : {VERSION}")
print(f"  RISK_PROFILE   : {RISK_PROFILE}")
print(f"  USE_SOFT_MASK  : {USE_SOFT_MASK}")
print(f"  RFR (annual)   : {ANNUAL_RISK_FREE_RATE:.4%}")
print(f"  Encoder        : {ENCODER_INPUT_DIM}→{ENCODER_HIDDEN_DIM}→{ENCODER_OUTPUT_DIM}")
print(f"  Attention heads: {ATTENTION_HEADS}")
print(f"  Policy head    : {POLICY_HIDDEN_DIMS}")

# =============================================================================
# PART 1 — FEATURE ENGINEERING
# =============================================================================

Configuration loaded.
  VERSION        : v1_initial
  RISK_PROFILE   : balanced
  USE_SOFT_MASK  : False
  RFR (annual)   : 6.4572%
  Encoder        : 10→64→64
  Attention heads: 1
  Policy head    : [256, 256]


In [21]:
feat_check = torch.load(TENSORS_DIR / "feature_tensor.pt", weights_only=True)
ret_check = torch.load(TENSORS_DIR / "returns_tensor.pt", weights_only=True)
mask_check = torch.load(TENSORS_DIR / "mask_tensor.pt", weights_only=True)
profile_check = torch.load(TENSORS_DIR / "profile_mask_tensor.pt", weights_only=True)
signal_check = torch.load(TENSORS_DIR / "market_signal.pt", weights_only=True)
splits_check = np.load(PREPROCESSING_DIR / "data_splits.npz")

T = feat_check.shape[0]

assert feat_check.ndim == 3
assert ret_check.shape == feat_check.shape[:2]
assert mask_check.shape == feat_check.shape[:2]
assert profile_check.shape == feat_check.shape[:2]
assert signal_check.shape == (T, 3)

assert np.isfinite(feat_check.numpy()).all()
assert np.isfinite(ret_check.numpy()).all()
assert np.isfinite(mask_check.numpy()).all()
assert np.isfinite(profile_check.numpy()).all()
assert np.isfinite(signal_check.numpy()).all()

_tr = splits_check["train"].tolist()
_va = splits_check["val"].tolist()
_te = splits_check["test"].tolist()

assert _tr[1] == _va[0]
assert _va[1] == _te[0]
assert _te[1] == T
assert len(splits_check["dates"]) == T

print("All sanity checks passed.")
print(f" feature_tensor  : {feat_check.shape} dtype={feat_check.dtype}")
print(f" returns_tensor  : {ret_check.shape}")
print(f" mask_tensor     : {mask_check.shape}")
print(f" profile_mask    : {profile_check.shape} coverage={profile_check.float().mean()*100:.1f}%")
print(f" market_signal   : {signal_check.shape}")
print(f" egx30 range     : [{signal_check[:,0].min():.3f}, {signal_check[:,0].max():.3f}]")
print(f" conia range     : [{signal_check[:,1].min():.3f}, {signal_check[:,1].max():.3f}]")
print(f" infl range      : [{signal_check[:,2].min():.3f}, {signal_check[:,2].max():.3f}]")
print(f" data_splits.npz  : {T} dates verified ✓")
print("\nPART 1 complete.")

All sanity checks passed.
 feature_tensor  : torch.Size([2647, 31, 9]) dtype=torch.float32
 returns_tensor  : torch.Size([2647, 31])
 mask_tensor     : torch.Size([2647, 31])
 profile_mask    : torch.Size([2647, 31]) coverage=31.1%
 market_signal   : torch.Size([2647, 3])
 egx30 range     : [-4.194, 3.423]
 conia range     : [-2.413, 6.619]
 infl range      : [-3.551, 3.231]
 data_splits.npz  : 2647 dates verified ✓

PART 1 complete.


In [22]:
feature_tensor      = torch.load(TENSORS_DIR / "feature_tensor.pt",      weights_only=False).numpy()
returns_tensor      = torch.load(TENSORS_DIR / "returns_tensor.pt",      weights_only=False).numpy()
mask_tensor         = torch.load(TENSORS_DIR / "mask_tensor.pt",         weights_only=False).numpy()
profile_mask_tensor = torch.load(TENSORS_DIR / "profile_mask_tensor.pt", weights_only=False).numpy()
market_signal       = torch.load(TENSORS_DIR / "market_signal.pt",       weights_only=False).numpy()

with open(PREPROCESSING_DIR / "market_signal_stats.json") as f:
    market_signal_stats = json.load(f)

_splits             = np.load(PREPROCESSING_DIR / "data_splits.npz")
date_index          = _splits["dates"].tolist()
_split_ranges       = {k: _splits[k].tolist() for k in ("train", "val", "test")}
global_split_indices = {k: list(range(*v)) for k, v in _split_ranges.items()}

# Reconstruct ticker list from OHLCV folder — no JSON needed
tickers          = sorted([p.stem.upper() for p in OHLCV_DIR.glob("*.csv")])
universe_indices = list(range(len(tickers)))
REL_VOL_IDX      = FEATURE_NAMES.index("rel_volume")
T_TOTAL          = feature_tensor.shape[0]
date_to_idx      = {d: i for i, d in enumerate(date_index)}

assert market_signal.shape[1] == 3
assert not np.isnan(market_signal).any()
assert T_TOTAL == len(date_index)

print(f"Artifacts loaded.")
print(f"  feature_tensor  : {feature_tensor.shape}")
print(f"  market_signal   : {market_signal.shape}")
print(f"  date_index      : {len(date_index)} dates  ({date_index[0]} → {date_index[-1]})")
print(f"  splits          : train {_split_ranges['train']}  "
      f"val {_split_ranges['val']}  test {_split_ranges['test']}")
print(f"  tickers ({len(tickers)})")

Artifacts loaded.
  feature_tensor  : (2647, 31, 9)
  market_signal   : (2647, 3)
  date_index      : 2647 dates  (2015-07-06 → 2026-06-04)
  splits          : train [0, 1579]  val [1579, 1823]  test [1823, 2647]
  tickers (31)


In [23]:
N_STOCKS     = len(tickers)
OBS_DIM      = N_STOCKS * ENCODER_INPUT_DIM + N_STOCKS + 3
FEATURES_DIM = N_STOCKS * ENCODER_OUTPUT_DIM + 3


class SharedStockEncoder(nn.Module):
    """
    Same MLP applied independently to every stock's 10-dim input vector
    (9 market features + 1 current weight). Shared weights prevent the
    network from learning ticker-specific patterns tied to input positions.

    Input : [batch, N, 10]
    Output: [batch, N, 64]
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, output_dim), nn.LayerNorm(output_dim), nn.Tanh(),
        )

    def forward(self, x):
        batch, N, D = x.shape
        return self.net(x.reshape(batch * N, D)).reshape(batch, N, -1)


class StockAttention(nn.Module):
    """
    Single-head self-attention over stock embeddings with profile mask.
    Ineligible stocks are excluded from the attention distribution before
    softmax — they contribute zero to any stock's context vector.

    Input : embeddings [batch, N, 64], profile_mask [batch, N]
    Output: [batch, N, 64]
    """
    def __init__(self, embed_dim, n_heads=1):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.embed_dim = embed_dim
        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.scale     = self.head_dim ** -0.5
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, embeddings, profile_mask):
        batch, N, D = embeddings.shape

        def split_heads(x):
            return x.reshape(batch, N, self.n_heads, self.head_dim).transpose(1, 2)

        Q, K, V = split_heads(self.q_proj(embeddings)), \
                  split_heads(self.k_proj(embeddings)), \
                  split_heads(self.v_proj(embeddings))

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if USE_SOFT_MASK:
            soft_bias = torch.log(profile_mask.clamp(min=1e-9)).unsqueeze(1).unsqueeze(2)
            scores = scores + soft_bias
        else:
            attn_mask = (profile_mask == 0).unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(attn_mask, float("-inf"))

        attn_weights = torch.nan_to_num(F_torch.softmax(scores, dim=-1), nan=0.0)
        out = torch.matmul(attn_weights, V).transpose(1, 2).reshape(batch, N, D)
        return embeddings + self.o_proj(out)   # residual connection


class AttentionExtractor(BaseFeaturesExtractor):
    """
    SB3 custom features extractor.

    Observation layout [OBS_DIM = N*10 + N + 3]:
      [0 : N*10]        per-stock features+weights
      [N*10 : N*10+N]   profile mask values (passed to attention)
      [N*10+N : +3]     macro signals

    Output [FEATURES_DIM = N*64 + 3]:
      attended stock embeddings flattened + macro signals
    """
    def __init__(self, observation_space, n_stocks, encoder_input_dim,
                 encoder_hidden_dim, encoder_output_dim, n_heads):
        super().__init__(observation_space, features_dim=n_stocks * encoder_output_dim + 3)
        self.n_stocks           = n_stocks
        self.encoder_input_dim  = encoder_input_dim
        self.encoder_output_dim = encoder_output_dim
        self.encoder   = SharedStockEncoder(encoder_input_dim, encoder_hidden_dim, encoder_output_dim)
        self.attention = StockAttention(encoder_output_dim, n_heads)

    def forward(self, obs):
        batch, N, F_in = obs.shape[0], self.n_stocks, self.encoder_input_dim
        stock_flat    = obs[:, :N * F_in]
        profile_flat  = obs[:, N * F_in : N * F_in + N]
        macro_signals = obs[:, N * F_in + N : N * F_in + N + 3]
        embeddings = self.encoder(stock_flat.reshape(batch, N, F_in))
        attended   = self.attention(embeddings, profile_flat)
        return torch.cat([attended.reshape(batch, N * self.encoder_output_dim), macro_signals], dim=1)


print(f"Architecture summary:")
print(f"  OBS_DIM      : {OBS_DIM}  ({N_STOCKS}×{ENCODER_INPUT_DIM} + {N_STOCKS} mask + 3 macro)")
print(f"  FEATURES_DIM : {FEATURES_DIM}  ({N_STOCKS}×{ENCODER_OUTPUT_DIM} + 3 macro)")

_dummy_obs = spaces.Box(low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32)
_ext = AttentionExtractor(_dummy_obs, N_STOCKS, ENCODER_INPUT_DIM, ENCODER_HIDDEN_DIM, ENCODER_OUTPUT_DIM, ATTENTION_HEADS)
print(f"  Extractor params: {sum(p.numel() for p in _ext.parameters()):,}")
del _dummy_obs, _ext

Architecture summary:
  OBS_DIM      : 344  (31×10 + 31 mask + 3 macro)
  FEATURES_DIM : 1987  (31×64 + 3 macro)
  Extractor params: 21,504


In [ ]:
class PortfolioEnv(gym.Env):
    """
    Observation layout [OBS_DIM]:
      [0 : N*10]       per-stock [9 features, current_weight] × N
      [N*10 : N*10+N]  profile mask values
      [N*10+N : +3]    market signals: egx30_ma_ratio, conia_delta, inflation_delta
    """
    def __init__(self, feature_tensor, returns_tensor, mask_tensor,
                 market_signal, profile_mask_tensor, universe_indices,
                 split_indices, risk_profile, feature_names,
                 mode="train", episode_length=24, rebalance_days=21, silent=False):
        super().__init__()
        self.feature_tensor      = feature_tensor
        self.returns_tensor      = returns_tensor
        self.mask_tensor         = mask_tensor
        self.market_signal       = market_signal
        self.profile_mask_tensor = profile_mask_tensor
        self.universe_indices    = universe_indices
        self.feature_names       = feature_names
        self.risk_profile        = risk_profile
        self.mode                = mode
        self.episode_length      = episode_length
        self.rebalance_days      = rebalance_days
        self.N           = len(universe_indices)
        self.F           = feature_tensor.shape[2]
        self.T           = feature_tensor.shape[0]
        self.reward_cfg  = REWARD_CONFIGS
        self.rel_vol_idx = REL_VOL_IDX

        rows_needed       = episode_length * rebalance_days
        self.valid_starts = [i for i in split_indices[mode] if i + rows_needed < self.T]

        obs_dim = self.N * (self.F + 1) + self.N + 3
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(low=-10.0, high=10.0, shape=(self.N + 1,), dtype=np.float32)

        if not silent:
            print(f"PortfolioEnv [{mode}] : {self.N} stocks | "
                  f"{len(self.valid_starts)} valid starts | obs=({obs_dim},)")

        self.current_step    = 0
        self.start_idx       = 0
        self.current_weights = np.zeros(self.N + 1, dtype=np.float32)
        self.portfolio_value = 1.0
        self.peak_value      = 1.0
        self.episode_returns = []

    def _get_combined_mask(self, day_idx):
        avail   = self.mask_tensor[day_idx, self.universe_indices]
        profile = self.profile_mask_tensor[day_idx, self.universe_indices]
        return (avail * profile).astype(np.float32) if USE_SOFT_MASK else \
               (avail * (profile > 0).astype(float)).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options is not None and "start_idx" in options:
            self.start_idx = options["start_idx"]
        elif self.mode == "train":
            self.start_idx = int(np.random.choice(self.valid_starts))
        else:
            self.start_idx = self.valid_starts[0]
        self.current_step    = 0
        self.portfolio_value = 1.0
        self.peak_value      = 1.0
        self.episode_returns = []
        self.current_weights = np.full(self.N + 1, 1.0 / (self.N + 1), dtype=np.float32)
        return self._get_observation(), {}

    def _get_observation(self):
        day_idx       = self.start_idx + self.current_step * self.rebalance_days
        combined_mask = self._get_combined_mask(day_idx)
        stock_features = self.feature_tensor[day_idx, self.universe_indices, :]
        stock_features = stock_features * (combined_mask > 0).astype(float)[:, np.newaxis]
        stock_with_weight = np.concatenate(
            [stock_features, self.current_weights[:-1].reshape(-1, 1)], axis=1
        ).astype(np.float32)
        return np.concatenate([
            stock_with_weight.flatten(),
            combined_mask,
            self.market_signal[day_idx].astype(np.float32),
        ]).astype(np.float32)

    def _apply_position_limits(self, weights, stock_mask):
        stock_weights = weights[:-1].copy()
        for _ in range(10):
            if not (stock_weights > MAX_STOCK_WEIGHT).any():
                break
            for i in range(len(stock_weights)):
                if stock_weights[i] > MAX_STOCK_WEIGHT:
                    excess = stock_weights[i] - MAX_STOCK_WEIGHT
                    stock_weights[i] = MAX_STOCK_WEIGHT
                    eligible = np.array([
                        j for j in range(len(stock_weights))
                        if j != i and stock_mask[j] > 0 and stock_weights[j] < MAX_STOCK_WEIGHT
                    ])
                    if len(eligible) > 0:
                        s = stock_weights[eligible].sum()
                        stock_weights[eligible] += excess * (stock_weights[eligible] / s) \
                                                   if s > 1e-8 else excess / len(eligible)
                    else:
                        weights[-1] += excess
        weights[:-1] = stock_weights
        return weights

    def _liquidity_adjusted_cost(self, day_idx, stock_turnover):
        rel_vol  = self.feature_tensor[day_idx, self.universe_indices, self.rel_vol_idx]
        liq_mult = np.clip(1.0 - 0.3 * rel_vol, 1.0, MAX_TRANSACTION_COST / BASE_TRANSACTION_COST)
        return float(np.dot(np.abs(stock_turnover), BASE_TRANSACTION_COST * liq_mult))

    def step(self, action):
        action      = np.array(action, dtype=np.float32)
        exp_action  = np.exp(action - action.max())
        raw_weights = exp_action / exp_action.sum()

        day_idx    = self.start_idx + self.current_step * self.rebalance_days
        stock_mask = self._get_combined_mask(day_idx)

        raw_weights[:-1] *= stock_mask if USE_SOFT_MASK else (stock_mask > 0).astype(float)
        s = raw_weights.sum()
        raw_weights = raw_weights / s if s > 1e-8 else \
                      np.array([0.0] * self.N + [1.0], dtype=np.float32)

        if raw_weights[-1] > MAX_CASH_WEIGHT:
            excess          = raw_weights[-1] - MAX_CASH_WEIGHT
            raw_weights[-1] = MAX_CASH_WEIGHT
            stock_sum       = raw_weights[:-1].sum()
            if stock_sum > 1e-8:
                raw_weights[:-1] += excess * (raw_weights[:-1] / stock_sum)
            else:
                valid_idx = np.where(stock_mask > 0)[0]
                if len(valid_idx) > 0:
                    raw_weights[valid_idx] += excess / len(valid_idx)
                else:
                    raw_weights[-1] = 1.0
            s = raw_weights.sum()
            if s > 1e-8: raw_weights = raw_weights / s

        raw_weights = self._apply_position_limits(raw_weights, stock_mask)
        s = raw_weights.sum()
        new_weights = raw_weights / s if s > 1e-8 else \
                      np.array([0.0] * self.N + [1.0], dtype=np.float32)

        proposed_turnover = float(np.sum(np.abs(new_weights - self.current_weights)))
        if proposed_turnover > TURNOVER_CAP:
            scale       = TURNOVER_CAP / proposed_turnover
            new_weights = self.current_weights + scale * (new_weights - self.current_weights)
            s           = new_weights.sum()
            if s > 1e-8: new_weights = new_weights / s

        stock_turnover   = new_weights[:-1] - self.current_weights[:-1]
        turnover         = float(np.sum(np.abs(stock_turnover)) + abs(new_weights[-1] - self.current_weights[-1]))
        transaction_cost = self._liquidity_adjusted_cost(day_idx, stock_turnover)

        eligible_mask    = (stock_mask > 0).astype(float)
        period_returns   = self.returns_tensor[day_idx:day_idx + self.rebalance_days, self.universe_indices]
        stock_ret_21d    = period_returns.sum(axis=0) * eligible_mask
        portfolio_return = float(np.dot(new_weights[:-1], stock_ret_21d))
        net_return       = portfolio_return - transaction_cost

        self.portfolio_value *= np.exp(net_return)
        self.peak_value       = max(self.peak_value, self.portfolio_value)
        self.episode_returns.append(net_return)

        drawdown = (self.peak_value - self.portfolio_value) / self.peak_value
        hhi      = float(np.sum(new_weights[:-1] ** 2))
        reward   = self.reward_cfg["return_weight"] * net_return
        reward  -= self.reward_cfg["turnover_penalty"] * turnover
        if self.reward_cfg["downside_penalty"] > 0 and net_return < 0:
            reward -= self.reward_cfg["downside_penalty"] * abs(net_return)
        reward -= self.reward_cfg["drawdown_penalty"] * drawdown
        reward -= self.reward_cfg["hhi_penalty"] * hhi
        regime_val = float(self.market_signal[day_idx, 0])
        if regime_val > 0 and self.reward_cfg.get("cash_penalty", 0) > 0:
            reward -= self.reward_cfg["cash_penalty"] * regime_val * max(0.0, new_weights[-1] - 0.10)

        self.current_weights = new_weights
        self.current_step   += 1
        truncated = self.current_step >= self.episode_length
        obs = self._get_observation() if not truncated else \
              np.zeros(self.observation_space.shape, dtype=np.float32)

        return obs, float(reward), False, truncated, {
            "portfolio_value": self.portfolio_value, "portfolio_return": portfolio_return,
            "net_return": net_return, "transaction_cost": transaction_cost,
            "turnover": turnover, "drawdown": drawdown, "hhi": hhi,
            "current_weights": new_weights.tolist(), "step": self.current_step,
        }

    def close(self): pass

In [25]:
def build_fold_split_indices(fold):
    dates_pd   = pd.to_datetime(date_index)
    train_mask = dates_pd <= fold["train_end"]
    val_mask   = (dates_pd > fold["train_end"]) & (dates_pd <= fold["val_end"])
    test_mask  = (dates_pd > fold["val_end"])   & (dates_pd <= fold["test_end"])
    train_idx  = list(np.where(train_mask)[0])
    val_idx    = list(np.where(val_mask)[0])
    test_idx   = list(np.where(test_mask)[0])
    print(f"  {fold['name']}: train {date_index[train_idx[0]]}→{date_index[train_idx[-1]]} "
          f"({len(train_idx)}d)  val ({len(val_idx)}d)  test ({len(test_idx)}d)")
    return {"train": train_idx, "val": val_idx, "test": test_idx}

print("Walk-forward folds:")
fold_split_indices = {fold["name"]: build_fold_split_indices(fold) for fold in WALK_FORWARD_FOLDS}

_fold1_train_days = len(fold_split_indices["fold_1"]["train"])
PERIOD_RISK_FREE_RATE = (1 + ANNUAL_RISK_FREE_RATE) ** (21 / 252) - 1
print(f"\nRFR: {ANNUAL_RISK_FREE_RATE:.4%}/yr → {PERIOD_RISK_FREE_RATE:.6%} per 21-day period")

Walk-forward folds:
  fold_1: train 2015-07-06→2019-12-31 (1092d)  val (243d)  test (244d)
  fold_2: train 2015-07-06→2020-12-31 (1335d)  val (244d)  test (244d)
  fold_3: train 2015-07-06→2021-12-30 (1579d)  val (244d)  test (242d)
  fold_4: train 2015-07-06→2022-12-29 (1823d)  val (242d)  test (484d)

RFR: 6.4572%/yr → 0.522806% per 21-day period


In [26]:
class RewardLoggerCallback(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path        = log_path
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_count   = 0

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
                self.episode_count += 1
                if self.verbose > 0 and self.episode_count % 200 == 0:
                    recent = self.episode_rewards[-200:]
                    sys.__stdout__.write(
                        f"  Episode {self.episode_count:>6} | mean (last 200): {np.mean(recent):>8.4f}\n"
                    )
                    sys.__stdout__.flush()
        return True

    def _on_training_end(self):
        if not self.episode_rewards:
            return
        pd.DataFrame({
            "episode":        range(1, len(self.episode_rewards) + 1),
            "episode_reward": self.episode_rewards,
            "episode_length": self.episode_lengths,
        }).to_csv(self.log_path, index=False)
        sys.__stdout__.write(
            f"Training history saved: {self.log_path}\n"
            f"Total episodes: {self.episode_count} | "
            f"Final mean (last 200): {np.mean(self.episode_rewards[-200:]):.4f}\n"
        )
        sys.__stdout__.flush()


class TrainingMetricsCallback(BaseCallback):
    def __init__(self, print_every_n_updates=20, verbose=1):
        super().__init__(verbose)
        self.print_every_n_updates = print_every_n_updates
        self.update_count          = 0

    def _on_rollout_end(self):
        self.update_count += 1
        if self.update_count % self.print_every_n_updates != 0:
            return True
        def gm(key):
            try: return self.model.logger.name_to_value.get(key)
            except: return None
        parts = [f"  [{self.num_timesteps:>7,} steps | update {self.update_count:>4}]"]
        for k, label in [("train/explained_variance", "expl_var"),
                         ("train/clip_fraction", "clip_frac"),
                         ("train/value_loss", "val_loss")]:
            v = gm(k)
            if v is not None: parts.append(f"{label}={v:>6.3f}")
        sys.__stdout__.write("  | ".join(parts) + "\n")
        sys.__stdout__.flush()
        return True

    def _on_step(self): return True


class EvalWithStopCallback(EvalCallback):
    def __init__(self, stop_callback, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stop_callback = stop_callback

    def _on_step(self):
        cont = super()._on_step()
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self.stop_callback.n_calls       = self.n_calls
            self.stop_callback.num_timesteps = self.num_timesteps
            self.stop_callback.parent        = self
            self.stop_callback.model         = self.model
            cont = cont and self.stop_callback._on_step()
        return cont

In [27]:
def compute_metrics(returns_arr, label, rfr=None, ann_rfr=None, print_results=True):
    rfr     = rfr     if rfr     is not None else PERIOD_RISK_FREE_RATE
    ann_rfr = ann_rfr if ann_rfr is not None else ANNUAL_RISK_FREE_RATE
    if len(returns_arr) == 0:
        return {}
    ann_return = returns_arr.mean() * PERIODS_PER_YEAR
    ann_vol    = returns_arr.std()  * np.sqrt(PERIODS_PER_YEAR)
    sharpe     = (ann_return - ann_rfr) / (ann_vol + 1e-8)
    excess     = returns_arr - rfr
    downside   = excess[excess < 0]
    sortino_v  = downside.std() * np.sqrt(PERIODS_PER_YEAR) if len(downside) > 0 else 1e-8
    sortino    = (ann_return - ann_rfr) / (sortino_v + 1e-8)
    cum        = np.exp(np.cumsum(returns_arr))
    peak       = np.maximum.accumulate(cum)
    max_dd     = ((peak - cum) / peak).max()
    calmar     = ann_return / max_dd if max_dd > 0.001 else np.nan
    var_95     = np.percentile(returns_arr, 5)
    cvar_95    = returns_arr[returns_arr <= var_95].mean() if (returns_arr <= var_95).any() else np.nan
    win_rate   = float((returns_arr > rfr).mean())

    if print_results:
        print(f"\n-- {label} --")
        print(f"  Ann. Return  : {ann_return*100:>8.2f}%  |  Ann. Vol    : {ann_vol*100:>8.2f}%")
        print(f"  Sharpe       : {sharpe:>8.3f}  |  Sortino     : {sortino:>8.3f}")
        print(f"  Max Drawdown : {max_dd*100:>8.2f}%  |  Calmar      : "
              f"{calmar:>8.3f}" if not np.isnan(calmar) else f"  Calmar       :      N/A")
        print(f"  CVaR (95%)   : {cvar_95*100:>7.2f}%  |  Win Rate RFR: {win_rate*100:>7.1f}%")

    return {"label": label, "n": len(returns_arr),
            "ann_return": ann_return, "ann_vol": ann_vol,
            "sharpe": sharpe, "sortino": sortino,
            "max_dd": max_dd, "calmar": calmar,
            "cvar_95": cvar_95, "win_rate": win_rate}


def block_bootstrap_ci(returns_arr, stat_fn, n_bootstrap=N_BOOTSTRAP,
                       block_size=BOOTSTRAP_BLOCK, ci=CONFIDENCE_LEVEL):
    n = len(returns_arr)
    if n < block_size * 2: return np.nan, np.nan
    n_blocks = int(np.ceil(n / block_size))
    boot_stats = []
    for _ in range(n_bootstrap):
        starts    = np.random.randint(0, n, size=n_blocks)
        resampled = np.concatenate([np.roll(returns_arr, -s)[:block_size] for s in starts])[:n]
        try:    boot_stats.append(stat_fn(resampled))
        except: continue
    boot_stats = np.array([s for s in boot_stats if np.isfinite(s)])
    if len(boot_stats) < 10: return np.nan, np.nan
    alpha = 1 - ci
    return np.percentile(boot_stats, 100*alpha/2), np.percentile(boot_stats, 100*(1-alpha/2))


def compute_bootstrap_cis(returns_arr, label, print_results=True):
    def sharpe_fn(r): return (r.mean()*PERIODS_PER_YEAR - ANNUAL_RISK_FREE_RATE) / (r.std()*np.sqrt(PERIODS_PER_YEAR) + 1e-8)
    def ret_fn(r):    return r.mean() * PERIODS_PER_YEAR
    def cvar_fn(r):
        v95  = np.percentile(r, 5)
        tail = r[r <= v95]
        return tail.mean() if len(tail) > 0 else np.nan

    sharpe_lo, sharpe_hi = block_bootstrap_ci(returns_arr, sharpe_fn)
    ret_lo,    ret_hi    = block_bootstrap_ci(returns_arr, ret_fn)
    cvar_lo,   cvar_hi   = block_bootstrap_ci(returns_arr, cvar_fn)

    if print_results:
        print(f"\n  Bootstrap CIs ({int(CONFIDENCE_LEVEL*100)}%) — {label}")
        print(f"    Ann. return: [{ret_lo*100:+.2f}%, {ret_hi*100:+.2f}%]")
        print(f"    Sharpe     : [{sharpe_lo:+.3f}, {sharpe_hi:+.3f}]")
        print(f"    CVaR 95%   : [{cvar_lo*100:+.2f}%, {cvar_hi*100:+.2f}%]")
    return {"sharpe_ci": (sharpe_lo, sharpe_hi), "ret_ci": (ret_lo, ret_hi), "cvar_ci": (cvar_lo, cvar_hi)}


def significance_test(rl_returns, eq_returns, label=""):
    diff    = (rl_returns - PERIOD_RISK_FREE_RATE) - (eq_returns - PERIOD_RISK_FREE_RATE)
    t_stat, t_pval = stats.ttest_1samp(diff, 0)
    try:    w_stat, w_pval = stats.wilcoxon(diff)
    except: w_stat, w_pval = np.nan, np.nan
    direction = "outperforms" if t_stat > 0 else "underperforms"
    print(f"  Significance ({label}): t={t_stat:+.3f} p={t_pval:.4f} "
          f"{'✓' if t_pval < 0.05 else '✗'} | W p={w_pval:.4f} "
          f"{'✓' if w_pval < 0.05 else '✗'} [{direction}]")
    return {"t_stat": t_stat, "t_pval": t_pval, "w_stat": w_stat, "w_pval": w_pval, "direction": direction}


def _make_eval_env(env_template, split_indices, mode, episode_length):
    return PortfolioEnv(
        feature_tensor=env_template.feature_tensor, returns_tensor=env_template.returns_tensor,
        mask_tensor=env_template.mask_tensor, market_signal=env_template.market_signal,
        profile_mask_tensor=env_template.profile_mask_tensor, universe_indices=env_template.universe_indices,
        split_indices=split_indices, risk_profile=env_template.risk_profile,
        feature_names=env_template.feature_names, mode=mode,
        episode_length=episode_length, rebalance_days=env_template.rebalance_days, silent=True,
    )


def multi_episode_evaluate(model, env_template, split_indices, mode, episode_length, max_episodes=MAX_EVAL_EPISODES):
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = eval_env.valid_starts
    if max_episodes and len(valid_starts) > max_episodes:
        valid_starts = [valid_starts[i] for i in np.linspace(0, len(valid_starts)-1, max_episodes, dtype=int)]
    all_returns, ep_metrics = [], []
    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        ep_ret, ep_val = [], 1.0
        for _ in range(episode_length):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            ep_ret.append(info["net_return"]); ep_val = info["portfolio_value"]
            if truncated: break
        all_returns.extend(ep_ret)
        ep_metrics.append({"start_idx": start_idx, "n_steps": len(ep_ret),
                            "final_value": ep_val, "mean_return": np.mean(ep_ret)})
    return np.array(all_returns), pd.DataFrame(ep_metrics)


def multi_episode_equal_weight(env_template, split_indices, mode, episode_length, max_episodes=MAX_EVAL_EPISODES):
    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = eval_env.valid_starts
    if max_episodes and len(valid_starts) > max_episodes:
        valid_starts = [valid_starts[i] for i in np.linspace(0, len(valid_starts)-1, max_episodes, dtype=int)]
    all_returns = []
    for start_idx in valid_starts:
        eq_prev_w, ep_ret = None, []
        for step in range(episode_length):
            day_idx = start_idx + step * env_template.rebalance_days
            if day_idx + env_template.rebalance_days >= env_template.T: break
            stock_mask = eval_env._get_combined_mask(day_idx)
            n_valid    = (stock_mask > 0).sum()
            eq_w       = (stock_mask > 0).astype(float) / n_valid if n_valid > 0 \
                         else np.zeros(len(env_template.universe_indices))
            turnover   = 1.0 if eq_prev_w is None else float(np.sum(np.abs(eq_w - eq_prev_w)))
            eq_prev_w  = eq_w.copy()
            period_rets = env_template.returns_tensor[day_idx:day_idx+env_template.rebalance_days,
                                                       env_template.universe_indices]
            gross = float(np.dot(eq_w, period_rets.sum(axis=0) * (stock_mask > 0).astype(float)))
            ep_ret.append(gross - BASE_TRANSACTION_COST * turnover)
        all_returns.extend(ep_ret)
    return np.array(all_returns)


def compute_active_metrics(rl_returns, eq_returns, model, env_template,
                           split_indices, mode, episode_length, fold_name,
                           max_episodes=MAX_EVAL_EPISODES, print_results=True):
    n        = min(len(rl_returns), len(eq_returns))
    active   = rl_returns[:n] - eq_returns[:n]
    ir       = float(active.mean() / (active.std() + 1e-8)) * np.sqrt(PERIODS_PER_YEAR)
    wr_vs_eq = float((rl_returns[:n] > eq_returns[:n]).mean())

    eval_env     = _make_eval_env(env_template, split_indices, mode, episode_length)
    valid_starts = eval_env.valid_starts
    if max_episodes and len(valid_starts) > max_episodes:
        valid_starts = [valid_starts[i] for i in np.linspace(0, len(valid_starts)-1, max_episodes, dtype=int)]

    all_turnovers, all_active_stocks = [], []
    for start_idx in valid_starts:
        obs, _ = eval_env.reset(options={"start_idx": start_idx})
        for _ in range(episode_length):
            day_idx    = eval_env.start_idx + eval_env.current_step * REBALANCE_DAYS
            stock_mask = eval_env._get_combined_mask(day_idx)
            all_active_stocks.append(int((stock_mask > 0).sum()))
            action, _ = model.predict(obs, deterministic=True)
            obs, _, _, truncated, info = eval_env.step(action)
            all_turnovers.append(info["turnover"])
            if truncated: break

    avg_turnover = float(np.mean(all_turnovers))
    avg_stocks   = float(np.mean(all_active_stocks))

    if print_results:
        print(f"\n-- Active Metrics [{fold_name} {mode}] --")
        print(f"  Information Ratio : {ir:>8.3f}")
        print(f"  Win Rate vs EQW   : {wr_vs_eq*100:>7.1f}%")
        print(f"  Avg Turnover/step : {avg_turnover*100:>7.1f}%")
        print(f"  Avg Active Stocks : {avg_stocks:>7.1f}")

    return {"information_ratio": ir, "win_rate_vs_eq": wr_vs_eq,
            "avg_turnover": avg_turnover, "avg_active_stocks": avg_stocks}

In [ ]:
_sanity_split = fold_split_indices["fold_1"]
sanity_env = PortfolioEnv(
    feature_tensor=feature_tensor, returns_tensor=returns_tensor,
    mask_tensor=mask_tensor, market_signal=market_signal,
    profile_mask_tensor=profile_mask_tensor, universe_indices=universe_indices,
    split_indices=_sanity_split, risk_profile=RISK_PROFILE,
    feature_names=FEATURE_NAMES, mode="train",
    episode_length=EPISODE_LENGTH, rebalance_days=REBALANCE_DAYS,
)

obs, _ = sanity_env.reset(seed=42)
expected_obs_dim = N_STOCKS * (len(FEATURE_NAMES) + 1) + N_STOCKS + 3

assert obs.shape[0] == expected_obs_dim == OBS_DIM, \
    f"Obs dim mismatch: {obs.shape[0]} vs {expected_obs_dim}"
assert not np.isnan(obs).any(), "NaN in observation"

macro_in_obs = obs[-3:]
assert np.allclose(macro_in_obs, market_signal[sanity_env.start_idx], atol=1e-5), "Macro mismatch"

profile_in_obs = obs[N_STOCKS * (len(FEATURE_NAMES)+1) : N_STOCKS * (len(FEATURE_NAMES)+1) + N_STOCKS]
day_mask = sanity_env._get_combined_mask(sanity_env.start_idx)
assert np.allclose(profile_in_obs, day_mask, atol=1e-5), "Profile mask mismatch"

for step in range(EPISODE_LENGTH):
    action = np.random.randn(sanity_env.action_space.shape[0]).astype(np.float32)
    obs, reward, _, truncated, info = sanity_env.step(action)
    assert not np.isnan(reward), f"NaN reward at step {step}"
    assert abs(sum(info["current_weights"]) - 1.0) < 1e-5
    assert info["current_weights"][-1] <= MAX_CASH_WEIGHT + 1e-5
    if truncated: break

_obs_space = spaces.Box(low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32)
_ext = AttentionExtractor(_obs_space, N_STOCKS, ENCODER_INPUT_DIM, ENCODER_HIDDEN_DIM, ENCODER_OUTPUT_DIM, ATTENTION_HEADS)
with torch.no_grad():
    _out = _ext(torch.from_numpy(obs).unsqueeze(0))
assert _out.shape == (1, FEATURES_DIM) and torch.isfinite(_out).all()
del _obs_space, _ext, _out

print(f"
Sanity check passed.")
print(f"  obs_dim        : {obs.shape[0]} ✓")
print(f"  Active stocks  : {int((day_mask > 0).sum())}/{N_STOCKS}")
print(f"  Episode steps  : {step+1}/{EPISODE_LENGTH} ✓")
print(f"  Extractor out  : (1, {FEATURES_DIM}) ✓")

PortfolioEnv [train] : 31 stocks | 1092 valid starts | obs=(344,)
✅ Sanity check passed.
  obs_dim        : 344 ✓
  Active stocks  : 9/31
  Episode steps  : 24/24 ✓
  Extractor out  : (1, 1987) ✓


In [ ]:
fold_models = {}

_template_env = PortfolioEnv(
    feature_tensor=feature_tensor, returns_tensor=returns_tensor,
    mask_tensor=mask_tensor, market_signal=market_signal,
    profile_mask_tensor=profile_mask_tensor, universe_indices=universe_indices,
    split_indices=fold_split_indices["fold_1"], risk_profile=RISK_PROFILE,
    feature_names=FEATURE_NAMES, mode="test",
    episode_length=EPISODE_LENGTH, rebalance_days=REBALANCE_DAYS, silent=True,
)

for fold in WALK_FORWARD_FOLDS:
    if fold["name"] != "fold_4":
        continue
    fold_name = fold["name"]
    split = fold_split_indices[fold_name]
    n_train_days   = len(split["train"])
    fold_timesteps = int(BASE_TIMESTEPS * n_train_days / _fold1_train_days)

    # All artifacts for this fold under model_versions/{VERSION}/{fold_name}/
    fold_dir = VERSION_DIR / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    # EvalCallback staging dir — renamed to versioned filename after training
    model_staging_dir = fold_dir / "_staging"
    model_staging_dir.mkdir(parents=True, exist_ok=True)
    final_model_path  = fold_dir / f"{VERSION}_{fold_name}.zip"

    print(f"\n{'='*62}")
    print(f"  TRAINING — {fold_name}  (→val {fold['val_end']})")
    print(f"  Train days : {n_train_days}  |  Timesteps : {fold_timesteps:,}")
    print(f"  Output dir : {fold_dir}")
    print(f"{'='*62}")

    def make_env_fn(s=split):
        def _init():
            return PortfolioEnv(
                feature_tensor=feature_tensor, returns_tensor=returns_tensor,
                mask_tensor=mask_tensor, market_signal=market_signal,
                profile_mask_tensor=profile_mask_tensor, universe_indices=universe_indices,
                split_indices=s, risk_profile=RISK_PROFILE,
                feature_names=FEATURE_NAMES, mode="train",
                episode_length=EPISODE_LENGTH, rebalance_days=REBALANCE_DAYS, silent=True,
            )
        return _init

    train_vec         = VecMonitor(DummyVecEnv([make_env_fn(split) for _ in range(N_ENVS)]))
    val_env_monitored = Monitor(PortfolioEnv(
        feature_tensor=feature_tensor, returns_tensor=returns_tensor,
        mask_tensor=mask_tensor, market_signal=market_signal,
        profile_mask_tensor=profile_mask_tensor, universe_indices=universe_indices,
        split_indices=split, risk_profile=RISK_PROFILE,
        feature_names=FEATURE_NAMES, mode="val",
        episode_length=EPISODE_LENGTH, rebalance_days=REBALANCE_DAYS, silent=True,
    ))

    policy_kwargs = dict(
        features_extractor_class  = AttentionExtractor,
        features_extractor_kwargs = dict(
            n_stocks           = N_STOCKS,
            encoder_input_dim  = ENCODER_INPUT_DIM,
            encoder_hidden_dim = ENCODER_HIDDEN_DIM,
            encoder_output_dim = ENCODER_OUTPUT_DIM,
            n_heads            = ATTENTION_HEADS,
        ),
        net_arch      = POLICY_HIDDEN_DIMS,
        activation_fn = nn.Tanh,
    )

    model = PPO(
        policy        = "MlpPolicy",
        env           = train_vec,
        seed          = 42,
        policy_kwargs = policy_kwargs,
        **PPO_PARAMS,
    )

    print(f"  Parameters : {sum(p.numel() for p in model.policy.parameters()):,}")

    stop_callback = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=50, min_evals=60, verbose=1,
    )
    eval_callback = EvalWithStopCallback(
        stop_callback        = stop_callback,
        eval_env             = val_env_monitored,
        best_model_save_path = str(model_staging_dir),
        # log_path + "evaluations.npz" = fold_dir/evaluations.npz (no subfolder)
        log_path             = str(fold_dir / "evaluations"),
        eval_freq            = max(20_000 // N_ENVS, 1),
        n_eval_episodes      = 1,
        deterministic        = True,
        render               = False,
        verbose              = 1,
    )
    reward_logger = RewardLoggerCallback(log_path=str(fold_dir / "training_history.csv"), verbose=1)
    metrics_cb    = TrainingMetricsCallback(print_every_n_updates=20, verbose=1)

    model.learn(
        total_timesteps     = fold_timesteps,
        callback            = [eval_callback, reward_logger, metrics_cb],
        progress_bar        = False,
        reset_num_timesteps = True,
    )

    # Rename staged best_model.zip → {VERSION}_{fold_name}.zip
    staged_model = model_staging_dir / "best_model.zip"
    if staged_model.exists():
        staged_model.rename(final_model_path)
        print(f"  Model saved: {final_model_path.name}")
    else:
        model.save(str(final_model_path.with_suffix("")))
        print(f"  Model saved (fallback): {final_model_path.name}")

    try:
        model_staging_dir.rmdir()   # remove staging dir if empty
    except OSError:
        pass

    best_model = PPO.load(str(final_model_path), env=val_env_monitored)
    fold_models[fold_name] = best_model
    print(f"  {fold_name} complete.")


  TRAINING — fold_4  (→val 2023-12-31)
  Train days : 1823  |  Timesteps : 5,842,948
  Output dir : C:\Users\mirae\Desktop\Personalization_Engine\model_versions\v1_initial\fold_4
  Parameters : 1,179,457
Eval num_timesteps=20000, episode_reward=-1.47 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=40000, episode_reward=-1.45 +/- 0.00
Episode length: 24.00 +/- 0.00
New best mean reward!
Eval num_timesteps=60000, episode_reward=-1.63 +/- 0.00
Episode length: 24.00 +/- 0.00


In [ ]:
all_fold_results = {}

for fold in WALK_FORWARD_FOLDS:
    fold_name = fold["name"]
    split     = fold_split_indices[fold_name]
    model     = fold_models[fold_name]

    print(f"\n{'='*60}")
    print(f"  EVALUATION — {fold_name}  (test → {fold['test_end']})")
    print(f"{'='*60}")

    rl_val_returns,  _ = multi_episode_evaluate(model, _template_env, split, "val",  EPISODE_LENGTH)
    eq_val_returns     = multi_episode_equal_weight(_template_env, split, "val",  EPISODE_LENGTH)
    rl_test_returns, _ = multi_episode_evaluate(model, _template_env, split, "test", EPISODE_LENGTH)
    eq_test_returns    = multi_episode_equal_weight(_template_env, split, "test", EPISODE_LENGTH)

    rl_val_metrics  = compute_metrics(rl_val_returns,  f"RL Agent — val [{fold_name}]")
    eq_val_metrics  = compute_metrics(eq_val_returns,   f"Equal Weight — val [{fold_name}]")
    rl_val_ci       = compute_bootstrap_cis(rl_val_returns,  f"RL val [{fold_name}]")
    val_sig         = significance_test(
        rl_val_returns[:min(len(rl_val_returns), len(eq_val_returns))],
        eq_val_returns[:min(len(rl_val_returns), len(eq_val_returns))],
        label=f"val {fold_name}"
    )

    rl_test_metrics = compute_metrics(rl_test_returns,  f"RL Agent — test [{fold_name}]")
    eq_test_metrics = compute_metrics(eq_test_returns,   f"Equal Weight — test [{fold_name}]")
    rl_test_ci      = compute_bootstrap_cis(rl_test_returns, f"RL test [{fold_name}]")
    test_sig        = significance_test(
        rl_test_returns[:min(len(rl_test_returns), len(eq_test_returns))],
        eq_test_returns[:min(len(rl_test_returns), len(eq_test_returns))],
        label=f"test {fold_name}"
    )

    active_metrics = compute_active_metrics(
        rl_returns=rl_test_returns, eq_returns=eq_test_returns,
        model=model, env_template=_template_env,
        split_indices=split, mode="test",
        episode_length=EPISODE_LENGTH, fold_name=fold_name,
    )

    fold_result = {
        "fold": fold_name, "version": VERSION,
        "train_end": fold["train_end"], "val_end": fold["val_end"], "test_end": fold["test_end"],
        "model_file": f"{VERSION}_{fold_name}.zip",
        "rl_val":  rl_val_metrics,  "eq_val":  eq_val_metrics,  "rl_val_ci":  rl_val_ci,
        "rl_test": rl_test_metrics, "eq_test": eq_test_metrics, "rl_test_ci": rl_test_ci,
        "val_significance": val_sig, "test_significance": test_sig,
        "active_metrics": active_metrics,
    }
    all_fold_results[fold_name] = fold_result

    # Save metrics.json directly in fold_dir
    with open(VERSION_DIR / fold_name / "metrics.json", "w") as f:
        json.dump(fold_result, f, indent=2, default=str)
    print(f"  Metrics saved: {VERSION_DIR / fold_name / 'metrics.json'}")

print(f"\n✅ All folds evaluated.")

In [ ]:
print(f"\n{'='*65}")
print(f"  CROSS-FOLD AGGREGATE — {VERSION}  ({len(WALK_FORWARD_FOLDS)} folds)")
print(f"{'='*65}")

metrics_keys = ["ann_return", "ann_vol", "sharpe", "sortino", "max_dd", "calmar", "cvar_95"]
rl_sharpes   = [all_fold_results[f]["rl_test"]["sharpe"]         for f in all_fold_results]
eq_sharpes   = [all_fold_results[f]["eq_test"]["sharpe"]         for f in all_fold_results]
rl_returns   = [all_fold_results[f]["rl_test"]["ann_return"]*100 for f in all_fold_results]
eq_returns   = [all_fold_results[f]["eq_test"]["ann_return"]*100 for f in all_fold_results]

print(f"\n{'Metric':<22} {'RL Mean':>10} {'RL Std':>8} | {'EQ Mean':>10} {'EQ Std':>8}")
print("-" * 67)
for k in metrics_keys:
    rl_vals = [all_fold_results[f]["rl_test"].get(k, np.nan) for f in all_fold_results]
    eq_vals = [all_fold_results[f]["eq_test"].get(k, np.nan) for f in all_fold_results]
    scale   = 100 if k in ["ann_return", "ann_vol", "max_dd", "cvar_95"] else 1
    unit    = "%" if scale == 100 else ""
    print(f"{k:<22} {np.nanmean(rl_vals)*scale:>9.2f}{unit} {np.nanstd(rl_vals)*scale:>7.2f}{unit} | "
          f"{np.nanmean(eq_vals)*scale:>9.2f}{unit} {np.nanstd(eq_vals)*scale:>7.2f}{unit}")

n_out = sum(
    1 for f in all_fold_results
    if all_fold_results[f]["test_significance"].get("t_pval", 1) < 0.05
    and all_fold_results[f]["test_significance"].get("direction") == "outperforms"
)
n_under = sum(
    1 for f in all_fold_results
    if all_fold_results[f]["test_significance"].get("t_pval", 1) < 0.05
    and all_fold_results[f]["test_significance"].get("direction") == "underperforms"
)
print(f"\nSignificantly outperforms EQW   : {n_out}/{len(WALK_FORWARD_FOLDS)} folds")
print(f"Significantly underperforms EQW : {n_under}/{len(WALK_FORWARD_FOLDS)} folds")

# Save aggregate_results.json directly in VERSION_DIR (no aggregate/ subfolder)
agg_summary = {
    "n_folds":         len(WALK_FORWARD_FOLDS),
    "version":         VERSION,
    "risk_profile":    RISK_PROFILE,
    "use_soft_mask":   USE_SOFT_MASK,
    "annual_rfr":      ANNUAL_RISK_FREE_RATE,
    "rrfr_source":     _rrfr_source,
    "architecture": {
        "encoder_input_dim":  ENCODER_INPUT_DIM,
        "encoder_hidden_dim": ENCODER_HIDDEN_DIM,
        "encoder_output_dim": ENCODER_OUTPUT_DIM,
        "attention_heads":    ATTENTION_HEADS,
        "policy_hidden_dims": POLICY_HIDDEN_DIMS,
        "obs_dim":            OBS_DIM,
        "features_dim":       FEATURES_DIM,
    },
    "folds":            [f["name"] for f in WALK_FORWARD_FOLDS],
    "rl_mean_sharpe":   float(np.mean(rl_sharpes)),
    "rl_std_sharpe":    float(np.std(rl_sharpes)),
    "eq_mean_sharpe":   float(np.mean(eq_sharpes)),
    "eq_std_sharpe":    float(np.std(eq_sharpes)),
    "sharpe_margin":    float(np.mean(rl_sharpes) - np.mean(eq_sharpes)),
    "rl_mean_return":   float(np.mean(rl_returns)),
    "eq_mean_return":   float(np.mean(eq_returns)),
    "n_outperforms":    int(n_out),
    "n_underperforms":  int(n_under),
}

with open(VERSION_DIR / "aggregate_results.json", "w") as f:
    json.dump(agg_summary, f, indent=2)

print(f"\n-- Final Summary --")
print(f"  VERSION         : {VERSION}")
print(f"  RFR             : {ANNUAL_RISK_FREE_RATE:.4%}  [{_rrfr_source}]")
print(f"  RL mean Sharpe  : {agg_summary['rl_mean_sharpe']:+.3f} ± {agg_summary['rl_std_sharpe']:.3f}")
print(f"  EQW mean Sharpe : {agg_summary['eq_mean_sharpe']:+.3f} ± {agg_summary['eq_std_sharpe']:.3f}")
print(f"  Sharpe margin   : {agg_summary['sharpe_margin']:+.3f}")
print(f"  RL mean return  : {agg_summary['rl_mean_return']:+.1f}%")
print(f"  EQW mean return : {agg_summary['eq_mean_return']:+.1f}%")
print(f"  Outperforms     : {n_out}/{len(WALK_FORWARD_FOLDS)} folds")
print(f"\n  Aggregate saved : {VERSION_DIR / 'aggregate_results.json'}")
print(f"\n✅ {VERSION} complete.")

In [ ]:
print("\n-- Signal/Cash Correlation Diagnostic --")
print("(Negative ρ = agent holds more cash in bearish regime ✓)")
for fold_name, model in fold_models.items():
    split    = fold_split_indices[fold_name]
    eval_env = PortfolioEnv(
        feature_tensor=feature_tensor, returns_tensor=returns_tensor,
        mask_tensor=mask_tensor, market_signal=market_signal,
        profile_mask_tensor=profile_mask_tensor, universe_indices=universe_indices,
        split_indices=split, risk_profile=RISK_PROFILE,
        feature_names=FEATURE_NAMES, mode="test",
        episode_length=EPISODE_LENGTH, rebalance_days=REBALANCE_DAYS, silent=True,
    )
    signals, cash_weights = [], []
    obs, _ = eval_env.reset()
    for _ in range(EPISODE_LENGTH):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, _, truncated, info = eval_env.step(action)
        day_idx = eval_env.start_idx + (eval_env.current_step - 1) * REBALANCE_DAYS
        signals.append(float(market_signal[day_idx, 0]))
        cash_weights.append(info["current_weights"][-1])
        if truncated: break
    corr   = np.corrcoef(signals, cash_weights)[0, 1]
    status = "✓ using signal" if corr < -0.1 else ("✗ not using signal" if corr > 0.1 else "~ weak")
    print(f"  {fold_name}: signal↔cash ρ = {corr:+.3f}  {status}")